# MNIST Dropout Experiment

Reproduction of the MNIST feed-forward experiment from Srivastava et al. (2014),
*Dropout: A Simple Way to Prevent Neural Networks from Overfitting*.

The same network is trained twice — **without dropout** and **with dropout**
(input dropout `p = 0.2`, hidden dropout `p = 0.5`) — and the two training curves are
compared.

| | |
|---|---|
| Architecture | `784 → 1024 → 1024 → 1024 → 10` (ReLU hidden units) |
| Data | MNIST, 20,000 train / 5,000 validation / 5,000 test |
| Optimizer | SGD, lr = 0.01, momentum = 0.95 |
| Batch size | 128 |
| Epochs | 20 |
| Loss | cross-entropy |

**How to run:** open this notebook in Google Colab, then *Runtime → Run all*.
The run takes a few minutes on a Colab GPU. The two figures are written to
`results/without_dropout.png` and `results/with_dropout.png`.

## 1. Device

In [ ]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 2. Dataset

MNIST is downloaded to `./data`. The 60,000 training images are split with a fixed
seed (42) into 20,000 train / 5,000 validation / 5,000 test, and the remaining
30,000 images are left unused so that the test split never overlaps training.

In [ ]:
transform = transforms.ToTensor()

mnist_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# We need:
# 20,000 training
# 5,000 validation
# 5,000 testing
# Total = 30,000

train_dataset, val_dataset, test_dataset, unused_dataset = random_split(
    mnist_dataset,
    [20000, 5000, 5000, 30000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

## 3. Neural Network

`784 → 1024 → 1024 → 1024 → 10`, ReLU on the three hidden layers. When dropout is
enabled, `nn.Dropout(0.2)` is applied to the input layer and `nn.Dropout(0.5)`
after each hidden activation; otherwise both are `nn.Identity()`. `nn.Dropout` uses
inverted dropout — survivors are scaled by `1/(1-p)` during training and nothing is
changed at test time.

In [ ]:
class Net(nn.Module):

    def __init__(self, dropout=False):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(784, 1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc3 = nn.Linear(1024, 1024)
        self.fc4 = nn.Linear(1024, 10)

        if dropout:
            self.input_dropout = nn.Dropout(0.2)
            self.hidden_dropout = nn.Dropout(0.5)
        else:
            self.input_dropout = nn.Identity()
            self.hidden_dropout = nn.Identity()


    def forward(self, x):

        x = self.flatten(x)

        x = self.input_dropout(x)

        x = torch.relu(self.fc1(x))
        x = self.hidden_dropout(x)

        x = torch.relu(self.fc2(x))
        x = self.hidden_dropout(x)

        x = torch.relu(self.fc3(x))
        x = self.hidden_dropout(x)

        x = self.fc4(x)

        return x

## 4. Accuracy Function

Accuracy is always measured with dropout switched off (`model.eval()`), so the two
runs are compared on equal terms.

In [ ]:
def get_accuracy(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    return 100 * correct / total

## 5. Training Function

One call trains a model for 20 epochs and returns the model, the per-epoch train and
validation accuracies, and the final accuracy on the held-out test split.

In [ ]:
def train_model(use_dropout):

    model = Net(dropout=use_dropout).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        model.parameters(),
        lr=0.01,
        momentum=0.95
    )

    train_accuracies = []
    val_accuracies = []

    for epoch in range(20):

        model.train()

        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total

        val_accuracy = get_accuracy(
            model,
            val_loader
        )

        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch {epoch + 1:2d} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Validation Acc: {val_accuracy:.2f}%"
        )

    # Final evaluation on test set
    test_accuracy = get_accuracy(
        model,
        test_loader
    )

    return model, train_accuracies, val_accuracies, test_accuracy

## 6. Without Dropout

In [ ]:
print("\n==============================")
print("WITHOUT DROPOUT")
print("==============================")

model_without, without_train_acc, without_val_acc, without_test_acc = train_model(False)

## 7. With Dropout

In [ ]:
print("\n==============================")
print("WITH DROPOUT")
print("==============================")

model_with, with_train_acc, with_val_acc, with_test_acc = train_model(True)

## 8. Final Results

In [ ]:
print("\n==============================")
print("FINAL RESULTS")
print("==============================")

print(
    f"Without Dropout Test Accuracy: "
    f"{without_test_acc:.2f}%"
)

print(
    f"With Dropout Test Accuracy: "
    f"{with_test_acc:.2f}%"
)

print(
    f"Without Dropout Test Error: "
    f"{100 - without_test_acc:.2f}%"
)

print(
    f"With Dropout Test Error: "
    f"{100 - with_test_acc:.2f}%"
)

## 9. Plot Without Dropout

Saved to `results/without_dropout.png`.

In [ ]:
os.makedirs("results", exist_ok=True)

epochs = range(1, 21)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    without_train_acc,
    'ro-',
    label="Train"
)

plt.plot(
    epochs,
    without_val_acc,
    'bo-',
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Without Dropout")

plt.legend()
plt.grid(True)

plt.savefig("results/without_dropout.png", dpi=150, bbox_inches="tight")

plt.show()

## 10. Plot With Dropout

Saved to `results/with_dropout.png`.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    with_train_acc,
    'ro-',
    label="Train"
)

plt.plot(
    epochs,
    with_val_acc,
    'bo-',
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("With Dropout")

plt.legend()
plt.grid(True)

plt.savefig("results/with_dropout.png", dpi=150, bbox_inches="tight")

plt.show()